# NPSC2002: Space Exploration -- Science, Technology and Industry
## Workshop 11: Resources Beyond Earth -- Asteroid Mining

---

**Learning objectives**

- Describe where near-Earth asteroids sit in the solar system relative to
  the main asteroid population
- Explain what makes an asteroid 'mission accessible' using delta-v data
- Interpret spectral type data to infer asteroid bulk composition,
  including water-bearing types
- Evaluate proposed extraction methods against the physical properties
  of target asteroids

**Data sources:** NASA/JPL NHATS API, NASA/JPL SBDB Query API,
curated meteorite analog composition data.
Both live APIs include synthetic fallback for offline use.


## Setup

Run this cell first.


In [ ]:
import subprocess, sys
for pkg in ['requests']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse, Wedge
import warnings
warnings.filterwarnings('ignore')

NAVY  = '#0B1E3E'
GOLD  = '#C8A951'
TEAL  = '#2E8B8B'
ROSE  = '#B5465A'
SAND  = '#D4A96A'
LIGHT = '#E8EEF6'
GREY  = '#7A8A9A'

plt.rcParams.update({
    'figure.facecolor': NAVY, 'axes.facecolor':   NAVY,
    'axes.edgecolor':   GOLD, 'axes.labelcolor':  GOLD,
    'xtick.color':      LIGHT,'ytick.color':      LIGHT,
    'text.color':       LIGHT,'grid.color':       '#1E3A5F',
    'grid.linestyle':   '--', 'grid.alpha':       0.5,
    'legend.facecolor': '#102040','legend.edgecolor': GOLD,
    'font.size': 11,
})
print('Setup complete.')


---
## Background: Where Do Near-Earth Asteroids Fit in the Solar System?

The vast majority of asteroids orbit in the **Main Belt** between Mars and Jupiter
(roughly 2.2 to 3.2 AU from the Sun). There are estimated to be several million
Main Belt objects larger than 1 km, and billions of smaller ones.

A small fraction of these have been gravitationally perturbed -- primarily by
Jupiter's resonances and collisions -- into orbits that bring them close to the
Sun and to Earth. These are **near-Earth asteroids (NEAs)**: asteroids with
perihelion distance less than 1.3 AU.

There are currently around 36,000 known NEAs. That sounds large, but it represents
less than 0.002% of the estimated total asteroid population.

**Potentially Hazardous Asteroids (PHAs)** are a subset of NEAs meeting two criteria:

- Absolute magnitude H <= 22 (roughly diameter >= 140 m)
- Minimum Orbit Intersection Distance (MOID) with Earth <= 0.05 AU (~7.5 million km)

MOID is the closest possible approach between two orbits, regardless of where
the bodies are at any given moment. A low MOID means that if orbital phases
align over long timescales, a close approach -- or impact -- becomes possible.
Around 2,300 PHAs are currently known. None is on a confirmed collision course.


### Solar system context

The diagram below places NEAs in context. Most asteroids are in the Main Belt.
NEAs are a dynamically perturbed minority with orbits that sweep through the
inner solar system, some crossing Earth's path.


In [ ]:
# ---- Solar system context diagram ----
# Shows the Main Belt population vs the NEA population spatially.
# NEA orbits are representative examples, not individually named.

def orbit_patch(a, e, theta_deg, **kwargs):
    """Ellipse patch for an orbit: Sun at one focus, perihelion at theta_deg."""
    b   = a * np.sqrt(max(1 - e**2, 1e-9))
    th  = np.radians(theta_deg)
    cx  = -a * e * np.cos(th)
    cy  = -a * e * np.sin(th)
    return Ellipse((cx, cy), 2*a, 2*b, angle=theta_deg, fill=False, **kwargs)

fig, ax = plt.subplots(figsize=(10, 9))
ax.set_aspect('equal')
ax.set_facecolor(NAVY)
ax.set_xlim(-4.0, 4.0)
ax.set_ylim(-4.0, 4.0)
ax.set_xlabel('Distance (AU)')
ax.set_ylabel('Distance (AU)')
ax.set_title(
    'Near-Earth Asteroids in the solar system context\n'
    '(representative NEA orbits shown -- not individual named objects)',
    color=GOLD, fontsize=12)

# ---- Main Belt: annular shading + random dots ----
belt = Wedge((0, 0), 3.2, 0, 360, width=3.2-2.2,
             facecolor='#263326', edgecolor=GREY,
             linewidth=0.5, alpha=0.7, zorder=1)
ax.add_patch(belt)

# Scatter random belt asteroids
rng = np.random.default_rng(12)
n_belt = 600
r_belt = rng.uniform(2.2, 3.2, n_belt)
theta_belt = rng.uniform(0, 2*np.pi, n_belt)
ax.scatter(r_belt * np.cos(theta_belt), r_belt * np.sin(theta_belt),
           s=1.5, color='#7AAA7A', alpha=0.5, zorder=2, linewidths=0)
ax.text(0, 2.85, 'Main Belt\n(millions of asteroids)',
        color='#9ACA9A', fontsize=9, ha='center', va='center')

# ---- Jupiter: dot + label only (outside plot range but indicate location) ----
ax.annotate('', xy=(3.85, 0), xytext=(3.4, 0),
            arrowprops=dict(arrowstyle='->', color=SAND, lw=1.2))
ax.text(3.6, 0.18, 'Jupiter\n5.2 AU', color=SAND, fontsize=8, ha='center')

# ---- Planetary orbit circles ----
for r, name, col, ls, lw in [
    (0.39, 'Mercury', GREY,      ':',  0.7),
    (0.72, 'Venus',   '#E8C84A', ':',  0.8),
    (1.00, 'Earth',   '#4A90D9', '-',  1.8),
    (1.52, 'Mars',    '#CD5C5C', '--', 1.0),
]:
    ax.add_patch(plt.Circle((0, 0), r, fill=False,
                            color=col, linestyle=ls, linewidth=lw, zorder=3))
    angle = np.radians(45)
    ax.text(r*np.cos(angle)+0.05, r*np.sin(angle)+0.05,
            name, color=col, fontsize=8, ha='left', va='bottom')

# ---- NEA perihelion zone: shading inside 1.3 AU ----
nea_zone = plt.Circle((0, 0), 1.3, fill=True,
                        facecolor='#1A2A3A', edgecolor=TEAL,
                        linewidth=1.0, linestyle=':', alpha=0.3, zorder=1)
ax.add_patch(nea_zone)

# ---- Representative NEA orbits ----
# Varied eccentricities and orientations; perihelion inside ~1.3 AU
nea_params = [
    # (a,   e,    theta)  -- some Earth-crossing, some just reaching inner belt
    (0.88, 0.52,  20),
    (1.45, 0.62, 100),
    (1.70, 0.70, 185),
    (2.10, 0.75, 255),
    (1.20, 0.42, 310),
    (0.95, 0.30, 145),
]
for i, (a, e, theta) in enumerate(nea_params):
    alpha = 0.55 + 0.1 * (i % 2)
    patch = orbit_patch(a, e, theta, edgecolor=TEAL,
                         linewidth=1.4, alpha=alpha, zorder=4)
    ax.add_patch(patch)

# ---- Labels ----
ax.plot([], [], color=TEAL, linewidth=1.6, label='Representative NEA orbits')
ax.add_patch(plt.Circle((0, 0), 0.05, color=TEAL, alpha=0.2))  # dummy for legend
ax.scatter([], [], color='#7AAA7A', s=8, label='Main Belt asteroids (sample)')
ax.add_patch(Wedge((0,0), 0.01, 0, 360, width=0, facecolor='#263326',
                   edgecolor=GREY, label='Main Belt region'))

# NEA zone label
ax.text(0, -1.15, 'NEA zone\n(q < 1.3 AU)', color=TEAL,
        fontsize=8, ha='center', alpha=0.8)

# ---- Sun ----
ax.plot(0, 0, 'o', color=GOLD, ms=16, zorder=6)
ax.text(0.10, 0.12, 'Sun', color=GOLD, fontsize=10)

ax.legend(loc='upper right', fontsize=8.5)
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()


**Key points from the diagram**

- The Main Belt (green region) is the dominant asteroid population.
  Objects there are gravitationally confined by Jupiter resonances.
- NEA orbits (teal ellipses) sweep through the inner solar system.
  Some reach out to the inner Main Belt at aphelion.
- An asteroid becomes an NEA when collisions or resonances change its orbit
  enough that its closest approach to the Sun (perihelion) falls inside 1.3 AU.
- NEAs are the closest asteroids to Earth in terms of both distance and
  delta-v -- which is why they are the focus of both resource missions
  and planetary defence.


---
## Part A: How Hard Is It to Reach an Asteroid?

### Delta-v and the NHATS list

**Delta-v** is the total velocity change a spacecraft must perform over a mission.
It sets the propellant requirement and, by the rocket equation, the overall mission cost.

For asteroid missions, delta-v is measured **from a 400 km circular low-Earth orbit (LEO)**.
Delta-v = 0 means you are already in LEO.
Every additional km/s requires exponentially more propellant.

The NHATS total delta-v includes four components:

1. Departure burn from LEO
2. Matching the asteroid's velocity at arrival
3. Departure burn from the asteroid
4. Any correction needed to control Earth atmospheric re-entry speed

NASA's NHATS list uses inclusive constraints (total delta-v <= 12 km/s,
duration <= 450 days) to map the full boundary of what is physically achievable.


### A1. Load NHATS data


In [ ]:
def load_nhats_live():
    url = 'https://ssd-api.jpl.nasa.gov/nhats.api'
    params = dict(dv=12, dur=450, stay=8, launch='2020-2045', h=26, occ=7)
    r = requests.get(url, params=params, timeout=15)
    r.raise_for_status()
    rows = []
    for item in r.json().get('data', []):
        rows.append({
            'designation': item.get('des', ''),
            'H':           float(item['h']) if item.get('h') else np.nan,
            'min_dv':      float(item['min_dv']['dv'])  if item.get('min_dv') else np.nan,
            'min_dv_dur':  float(item['min_dv']['dur']) if item.get('min_dv') else np.nan,
            'n_traj':      int(item['n_via_traj']) if item.get('n_via_traj') else 0,
            'min_size_m':  float(item['min_size']) if item.get('min_size') else np.nan,
            'max_size_m':  float(item['max_size']) if item.get('max_size') else np.nan,
        })
    df = pd.DataFrame(rows)
    df['mid_size_m'] = (df['min_size_m'] + df['max_size_m']) / 2
    return df


def load_nhats_synthetic():
    rng = np.random.default_rng(42)
    n = 1200
    min_dv = np.clip(rng.lognormal(np.log(7.5), 0.35, n), 3.5, 12.0)
    H      = rng.uniform(17, 27, n)
    albedo = rng.choice([0.05, 0.10, 0.20, 0.35], n, p=[0.15, 0.30, 0.35, 0.20])
    diam   = 1329 / np.sqrt(albedo) * 10 ** (-H / 5) * 1000
    return pd.DataFrame({
        'designation': [f'SYNTH-{i:04d}' for i in range(n)],
        'H':           np.round(H, 1),
        'min_dv':      np.round(min_dv, 2),
        'min_dv_dur':  np.round(rng.uniform(180, 440, n), 0),
        'n_traj':      (rng.exponential(80000, n)*(12-min_dv)/8).astype(int).clip(1),
        'min_size_m':  np.round(diam * 0.4, 0),
        'max_size_m':  np.round(diam * 2.5, 0),
        'mid_size_m':  np.round(diam, 0),
    })


try:
    nhats = load_nhats_live()
    print(f'Live NHATS data: {len(nhats)} mission-accessible NEAs.')
    NHATS_SOURCE = 'NASA NHATS API (live)'
except Exception as e:
    print(f'Live API unavailable ({e}). Using synthetic data.')
    nhats = load_nhats_synthetic()
    NHATS_SOURCE = 'Synthetic (NHATS-representative)'

print(nhats.columns.tolist())
nhats.head()


### A2. Delta-v distribution


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(nhats['min_dv'].dropna(), bins=40,
        color=GOLD, edgecolor=NAVY, linewidth=0.5)

# Reference values -- all measured from 400 km LEO (delta-v = 0 at LEO)
refs = [
    (3.2, 'GTO transfer (~3.2 km/s)',   TEAL),
    (6.0, 'Practical target threshold',  ROSE),
    (8.7, 'Lunar surface (~8.7 km/s)',   SAND),
]
ymax = ax.get_ylim()[1]
for dv, label, col in refs:
    ax.axvline(dv, color=col, linestyle='--', linewidth=1.4)
    ax.text(dv + 0.12, ymax * 0.88, label, color=col, fontsize=8.5, va='top')

ax.set_xlabel('Minimum round-trip delta-v from LEO (km/s)')
ax.set_ylabel('Number of asteroids')
ax.set_title(
    f'NHATS-accessible NEAs: delta-v distribution\n'
    f'Reference point: 400 km circular LEO (delta-v = 0 km/s baseline)   |   {NHATS_SOURCE}',
    color=GOLD, fontsize=11)
ax.grid(True)
plt.tight_layout()
plt.show()


### A3. Estimated diameter vs delta-v


In [ ]:
plot_df = nhats.dropna(subset=['min_dv', 'mid_size_m']).copy()
plot_df = plot_df[plot_df['mid_size_m'] < 8000]

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(plot_df['min_dv'], plot_df['mid_size_m'],
           color=GOLD, s=14, alpha=0.55, linewidths=0)
ax.axvline(6.0, color=ROSE, linestyle='--', linewidth=1.2,
           label='Practical threshold (6 km/s from LEO)')
ax.set_xlabel('Minimum round-trip delta-v from LEO (km/s)')
ax.set_ylabel('Estimated diameter (m, log scale)')
ax.set_title(
    f'Estimated diameter vs accessibility\n'
    f'Size derived from H magnitude and assumed albedo   |   {NHATS_SOURCE}',
    color=GOLD, fontsize=11)
ax.set_yscale('log')
ax.legend(fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()


### Activity: Explore the NHATS data yourself

The plots above show the overall distribution.
Now query the DataFrame directly to find specific answers.

Replace each `???` with appropriate values or column names.
The column names are printed in A1 -- refer to that output.


In [ ]:
# ---- Part 1 ----
# The lunar surface requires roughly 8.7 km/s total delta-v from LEO.
# How many NHATS targets are MORE accessible than the Moon?
# Set dv_limit to the lunar surface value (check the reference lines in A2).

dv_limit = ???    # km/s

sub = nhats[nhats['min_dv'] <= dv_limit]
print(f'Targets within {dv_limit} km/s of LEO: {len(sub)}')
print(f'That is {100*len(sub)/len(nhats):.1f}% of all NHATS targets.')


# ---- Part 2 ----
# Find the five easiest-to-reach targets.
# Modify sort_values() to sort by the correct column, lowest first.

top5 = nhats.sort_values(???, ascending=True).head(5)
print()
print('Five most accessible NHATS targets:')
print(top5[['designation', 'min_dv', 'min_dv_dur', 'mid_size_m']].to_string(index=False))


# ---- Part 3 ----
# What is the median mission duration for targets with delta-v <= 6 km/s?
# Complete the column name inside the square brackets.

median_dur = nhats[nhats['min_dv'] <= 6][???].median()
print()
print(f'Median mission duration (dv <= 6 km/s): {median_dur:.0f} days')


---
### Question

Part 3 gives you the median mission duration for the most accessible targets.
A crew mission to the International Space Station lasts around 180 days.
A crewed Mars mission is estimated at around 900 days.
Where does a typical asteroid mission sit relative to those benchmarks,
and what does that imply for the crew health and life support requirements?

---


### Your answer

*(Write your response here.)*


### A4. The flip side: accessibility and hazard

Low delta-v arises from an orbit very similar to Earth's.
That same property can also produce a small MOID -- meaning the asteroid
could come very close to Earth over long timescales.

The cell below cross-references NHATS targets with MOID and PHA data from SBDB.


In [ ]:
def load_neo_moid_live():
    url = 'https://ssd-api.jpl.nasa.gov/sbdb_query.api'
    params = {
        'fields':   'pdes,pha,class,H,moid',
        'sb-class': 'IEO,ATE,APO,AMO',
        'limit':    3000,
    }
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    payload = r.json()
    df = pd.DataFrame(payload['data'], columns=payload['fields'])
    df['moid'] = pd.to_numeric(df['moid'], errors='coerce')
    df['H']    = pd.to_numeric(df['H'],    errors='coerce')
    df['pha']  = df['pha'].map({'Y': True, 'N': False}).fillna(False)
    return df


def load_neo_moid_synthetic():
    rng = np.random.default_rng(99)
    n   = 2500
    moid = np.clip(rng.lognormal(-1.5, 1.2, n), 0.001, 0.8)
    H    = rng.normal(21, 3.5, n).clip(14, 30)
    pha  = (moid <= 0.05) & (H <= 22)
    return pd.DataFrame({
        'pdes':  [f'SYNTH-{i:05d}' for i in range(n)],
        'pha':   pha,
        'class': rng.choice(['APO','AMO','ATE','IEO'], n, p=[0.55,0.33,0.11,0.01]),
        'H':     np.round(H, 1),
        'moid':  np.round(moid, 5),
    })


try:
    neo_moid = load_neo_moid_live()
    print(f'Live SBDB MOID data: {len(neo_moid)} NEAs.')
    MOID_SOURCE = 'NASA JPL SBDB (live)'
except Exception as e:
    print(f'Live API unavailable ({e}). Using synthetic data.')
    neo_moid = load_neo_moid_synthetic()
    MOID_SOURCE = 'Synthetic'

# Cross-reference with NHATS
nhats_moid = nhats.merge(
    neo_moid.rename(columns={'pdes': 'designation'})[['designation','pha','moid']],
    on='designation', how='left'
)

n_matched = nhats_moid['moid'].notna().sum()
n_pha     = nhats_moid['pha'].sum()
print(f'NHATS targets matched to SBDB MOID data: {n_matched} of {len(nhats_moid)}')
print(f'Of matched targets, classified as PHAs:  {int(n_pha)}')
print()
print('PHAs in NHATS by delta-v bin:')
for lo, hi in [(3,5),(5,7),(7,9),(9,12)]:
    sub = nhats_moid[(nhats_moid['min_dv']>=lo)&(nhats_moid['min_dv']<hi)]
    p   = sub['pha'].sum()
    if len(sub):
        print(f'  dv {lo}-{hi} km/s: {len(sub):>4} targets, {int(p):>3} PHAs ({100*p/len(sub):.0f}%)')


In [ ]:
# MOID vs delta-v scatter, coloured by PHA status
pm = nhats_moid.dropna(subset=['min_dv','moid'])
pha_m    = pm['pha'] == True
nonpha_m = ~pha_m

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(pm.loc[nonpha_m,'min_dv'], pm.loc[nonpha_m,'moid'],
           color=GOLD, s=14, alpha=0.45, linewidths=0, label='Non-PHA')
ax.scatter(pm.loc[pha_m,'min_dv'],    pm.loc[pha_m,'moid'],
           color=ROSE, s=22, alpha=0.85, linewidths=0,
           label='PHA  (MOID <= 0.05 AU and H <= 22)')
ax.axhline(0.05, color=ROSE, linestyle='--', linewidth=1.2,
           label='PHA MOID threshold (0.05 AU)')
ax.axvline(6.0, color=TEAL, linestyle='--', linewidth=1.2,
           label='Practical mission threshold (6 km/s)')
ax.set_xlabel('Minimum round-trip delta-v from LEO (km/s)')
ax.set_ylabel('MOID with Earth (AU)')
ax.set_title(
    f'Accessibility vs hazard: MOID vs delta-v for NHATS targets\n'
    f'NHATS: {NHATS_SOURCE}   |   MOID: {MOID_SOURCE}',
    color=GOLD, fontsize=11)
ax.set_ylim(-0.01, 0.45)
ax.legend(fontsize=8.5)
ax.grid(True)
plt.tight_layout()
plt.show()

print('Lower delta-v corresponds to a more Earth-like orbit,')
print('which also tends to produce a lower MOID.')
print('The easiest mining targets overlap with the most carefully monitored PHAs.')


### A5. Should some asteroids be off-limits?

The scatter above shows that a subset of the most accessible mining targets
are also PHAs -- objects we monitor for potential future Earth impact.

This matters for resource missions for several reasons:

- Mining changes an asteroid's mass distribution, surface albedo, and centre of mass.
  All of these affect long-term orbital evolution through radiation pressure (Yarkovsky effect)
  and gravitational perturbations.
- Even small unplanned impulses -- docking manoeuvres, thruster exhaust, mass ejection --
  can alter a trajectory measurably over decades.
- The 2022 DART mission demonstrated that a deliberate kinetic impactor shifts
  an asteroid's orbit. Mining infrastructure is, in effect, an unplanned deflection system.
- Conversely, a PHA with established mining infrastructure could also be *deliberately*
  deflected using that same equipment -- turning a hazard into an asset.
- No existing international treaty addresses unintentional orbit modification
  by a commercial resource mission. The 1967 Outer Space Treaty was not written
  with asteroid mining in mind.


---
### Question

The scatter plot shows a cluster of asteroids with low delta-v AND low MOID:
easy to reach, and potentially hazardous.

Should such objects be available for commercial mining?
Consider: what information you would need before committing,
who would have authority to approve or veto such a mission,
and whether the mining infrastructure changes the risk picture
for better or worse.

---


### Your answer

*(Write your response here.)*


---
## Part B: What Are Accessible Asteroids Made Of?

### Background: spectral taxonomy and bulk composition

Astronomers classify asteroids by their **reflectance spectra**.
The dominant modern system, Bus-DeMeo (2009), defines 25 subtypes.
These map onto bulk compositions through comparison with meteorites measured in the lab.

| Class | Albedo | Meteorite analog | Key resources |
|---|---|---|---|
| **C-complex** (C, B, Ch, Cb...) | Dark (0.03--0.10) | CI/CM carbonaceous chondrites | **Water** (OH in phyllosilicates), organic compounds |
| **S-complex** (S, Q, Sq, Sr...) | Moderate (0.10--0.30) | Ordinary chondrites, stony-iron | Iron-nickel metal, silicates, platinum-group elements |
| **M-type** (M, Xe...) | Moderate-high (0.10--0.30) | Iron meteorites | Bulk iron-nickel; highest potential PGM concentrations |
| **Other** (V, D, A, L, K...) | Variable | Various | Composition often uncertain |

**Why C-type water matters**

C and B-type asteroids contain phyllosilicates -- clay-like minerals where
water is chemically bonded as hydroxyl (OH). Heating to 200--400 degrees Celsius
drives off water vapour, which can then be split by electrolysis into
H2 (fuel) and O2 (oxidiser), used for life support, or used as radiation shielding.

The 2020 Hayabusa2 sample return from C-type Ryugu confirmed organic compounds,
hydrated silicates, and amino acids -- matching and exceeding spectral predictions.

**The characterisation problem:** spectral data exists for only a few thousand
of the ~36,000 known NEAs, biased strongly toward larger, brighter objects.
The composition of most mission-accessible targets is unknown.


### B1. Load NEO spectral data from SBDB


In [ ]:
def load_sbdb_spectral_live():
    url = 'https://ssd-api.jpl.nasa.gov/sbdb_query.api'
    params = {
        'fields':   'pdes,class,H,diameter,albedo,spec_B,spec_T',
        'sb-class': 'IEO,ATE,APO,AMO',
        'limit':    500,
    }
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    payload = r.json()
    df = pd.DataFrame(payload['data'], columns=payload['fields'])
    df['spec'] = df['spec_B'].combine_first(df['spec_T'])
    df = df[df['spec'].notna() & (df['spec'] != '')].copy()
    for col in ['H', 'diameter', 'albedo']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def load_sbdb_spectral_synthetic():
    rng = np.random.default_rng(7)
    n = 1000
    type_pool = (
        ['S']*34 + ['Q']*20 + ['C']*12 + ['X']*8 +
        ['V']*7  + ['B']*5  + ['D']*4  + ['A']*3 +
        ['L']*3  + ['K']*2  + ['M']*2
    )
    spec   = rng.choice(type_pool, n)
    H      = rng.normal(19, 3, n).clip(14, 27)
    amap   = {'S':0.20,'Q':0.22,'C':0.06,'X':0.10,'V':0.35,
               'B':0.07,'D':0.04,'A':0.25,'L':0.13,'K':0.14,'M':0.18}
    albedo = np.array([amap.get(s, 0.12) for s in spec])
    albedo = (albedo + rng.normal(0, 0.02, n)).clip(0.02, 0.60)
    diam   = 1329 / np.sqrt(albedo) * 10 ** (-H / 5)
    return pd.DataFrame({
        'pdes':     [f'SYNTH-{i:05d}' for i in range(n)],
        'spec':     spec,
        'H':        np.round(H, 1),
        'diameter': np.round(diam, 3),
        'albedo':   np.round(albedo, 3),
        'class':    rng.choice(['APO','AMO','ATE'], n, p=[0.60,0.30,0.10]),
    })


try:
    sbdb = load_sbdb_spectral_live()
    print(f'Live SBDB spectral data: {len(sbdb)} NEAs with spectral type.')
    SPEC_SOURCE = 'NASA JPL SBDB (live)'
except Exception as e:
    print(f'Live API unavailable ({e}). Using synthetic data.')
    sbdb = load_sbdb_spectral_synthetic()
    SPEC_SOURCE = 'Synthetic (DeMeo & Carry 2014 distribution)'

sbdb.head()


### B2. Resource categories


In [ ]:
C_COMPLEX = {'C','B','Cb','Ch','Cg','Cgh','Xc','Xk'}
S_COMPLEX = {'S','Q','Sq','Sr','Sa','Sl','Sw','Sv'}
M_TYPE    = {'M','X','Xe'}

def categorise(spec):
    if not isinstance(spec, str):
        return 'Unknown'
    s = spec.strip()
    if s in C_COMPLEX or s.startswith('C') or s.startswith('B'):
        return 'C-complex (water / organics)'
    if s in S_COMPLEX or s.startswith('S') or s.startswith('Q'):
        return 'S-complex (silicate / metal)'
    if s in M_TYPE or s.startswith('M'):
        return 'M-type (metallic)'
    return 'Other (composition uncertain)'

sbdb['resource_class'] = sbdb['spec'].apply(categorise)
counts = sbdb['resource_class'].value_counts()

print('Spectral breakdown (characterised NEOs only):')
for cat, n in counts.items():
    print(f'  {cat:<38} {n:>5}  ({100*n/len(sbdb):.1f}%)')
print()
print('C-types are darker -- underrepresented in observation surveys.')
print('The true NEO population has a higher C-type fraction than this sample.')


### B3. Spectral type distribution


In [ ]:
RESOURCE_COLOURS = {
    'C-complex (water / organics)': TEAL,
    'S-complex (silicate / metal)': GOLD,
    'M-type (metallic)':            ROSE,
    'Other (composition uncertain)': SAND,
    'Unknown':                       LIGHT,
}

fig, ax = plt.subplots(figsize=(7, 6))
labels  = counts.index.tolist()
colours = [RESOURCE_COLOURS.get(l, LIGHT) for l in labels]

wedges, _, autotexts = ax.pie(
    counts.values, labels=None, colors=colours,
    autopct='%1.1f%%', startangle=140,
    wedgeprops={'edgecolor': NAVY, 'linewidth': 1.5},
    textprops={'color': LIGHT}
)
for at in autotexts:
    at.set_fontsize(9)
ax.legend(wedges, labels, loc='lower left', fontsize=8.5,
          title='Resource class', title_fontsize=9)
ax.set_title(f'Spectral classes: characterised NEOs only\n({SPEC_SOURCE})',
             color=GOLD, fontsize=11)
plt.tight_layout()
plt.show()


### B4. Bulk composition: S-type vs C-type

Spectral type is a proxy. The chart below shows actual bulk mineralogy,
from laboratory analysis of meteorite analogs.

- **S-type analog:** LL ordinary chondrite (Dunn et al. 2010)
  -- similar to Hayabusa samples from (25143) Itokawa
- **C-type analog:** CI carbonaceous chondrite (Lodders & Fegley 1998;
  Zolensky et al. 2018) -- similar to Hayabusa2 samples from (162173) Ryugu


In [ ]:
categories = [
    'Olivine',
    'Pyroxene',
    'Iron-nickel metal',
    'Feldspar',
    'Troilite (FeS)',
    'Phyllosilicates\n(hydrated clays)',
    'Water (OH-bound)',
    'Organic compounds',
    'Other minerals',
]
s_type = [38.0, 26.0, 17.0,  9.0,  6.0,  0.5,  0.1,  0.3,  3.1]
c_type = [12.0,  4.0,  1.0,  0.5,  5.0, 52.0, 10.0,  5.5, 10.0]

highlight = {
    'Water (OH-bound)':           TEAL,
    'Phyllosilicates\n(hydrated clays)': '#5ECECE',
    'Iron-nickel metal':          ROSE,
    'Organic compounds':          SAND,
}
s_cols = [highlight.get(k, '#3A5A8A') for k in categories]
c_cols = [highlight.get(k, '#2A6A4A') for k in categories]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
y, h = np.arange(len(categories)), 0.65

bars_s = ax1.barh(y, s_type, height=h, color=s_cols, edgecolor=NAVY, linewidth=0.5)
ax1.set_yticks(y)
ax1.set_yticklabels(categories, fontsize=9.5)
ax1.set_xlabel('Mass percent (%)')
ax1.set_title('S-type (LL ordinary chondrite analog)\ne.g. (25143) Itokawa',
              color=GOLD, fontsize=10.5)
ax1.set_xlim(0, 58)
ax1.grid(True, axis='x')
for bar, val in zip(bars_s, s_type):
    if val >= 0.5:
        ax1.text(val+0.5, bar.get_y()+h/2, f'{val:.1f}%',
                 va='center', fontsize=8.5, color=LIGHT)

bars_c = ax2.barh(y, c_type, height=h, color=c_cols, edgecolor=NAVY, linewidth=0.5)
ax2.set_xlabel('Mass percent (%)')
ax2.set_title('C-type (CI carbonaceous chondrite analog)\ne.g. (162173) Ryugu',
              color=GOLD, fontsize=10.5)
ax2.set_xlim(0, 65)
ax2.grid(True, axis='x')
for bar, val in zip(bars_c, c_type):
    if val >= 0.5:
        ax2.text(val+0.5, bar.get_y()+h/2, f'{val:.1f}%',
                 va='center', fontsize=8.5, color=LIGHT)

legend_patches = [
    mpatches.Patch(color=TEAL,      label='Water (OH-bearing)'),
    mpatches.Patch(color='#5ECECE', label='Phyllosilicates (hydrated silicates)'),
    mpatches.Patch(color=ROSE,      label='Iron-nickel metal'),
    mpatches.Patch(color=SAND,      label='Organic compounds'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=2,
           fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, -0.06))
plt.suptitle(
    'Bulk mineral composition: S-type vs C-type asteroid analogs\n'
    '(Sources: Dunn et al. 2010; Lodders & Fegley 1998; Zolensky et al. 2018)',
    color=LIGHT, fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print('Key contrasts:')
print(f'  Iron-nickel:       S-type {s_type[2]:.0f} wt%   vs C-type {c_type[2]:.0f} wt%')
print(f'  Water (OH-bound):  S-type {s_type[6]:.1f} wt%  vs C-type {c_type[6]:.0f} wt%')
print(f'  Phyllosilicates:   S-type {s_type[5]:.1f} wt%  vs C-type {c_type[5]:.0f} wt%')
print('  Phyllosilicates are the primary water host -- the thermal extraction target.')


### B5. The characterisation gap


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bins = np.arange(14, 28, 0.5)
ax.hist(nhats['H'].dropna(), bins=bins, color=LIGHT, alpha=0.5,
        label='All NHATS targets', edgecolor=NAVY)
ax.hist(sbdb['H'].dropna(), bins=bins, color=GOLD, alpha=0.75,
        label='Spectrally characterised NEOs', edgecolor=NAVY)
ax.set_xlabel('Absolute magnitude H  (smaller H = larger / brighter)')
ax.set_ylabel('Count')
ax.set_title('The characterisation gap:\nspectral data is biased toward large bright objects',
             color=GOLD, fontsize=11)
ax.legend(fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()

if 'pdes' in sbdb.columns:
    n_match = nhats['designation'].isin(sbdb['pdes']).sum()
    print(f'NHATS targets with known spectral type: {n_match} of {len(nhats)}')


---
### Question

The composition plots show that S-types and C-types are fundamentally different materials.
S-types are more common among accessible NEAs, but C-types carry water.
If sustaining deep-space human presence is the goal, which type would you prioritise
for a first resource mission -- and why?
What additional information would change your answer?

---


### Your answer

*(Write your response here.)*


---
## Part C: How Would You Extract the Resource?

### Background

Identifying a resource is only the first step.
Extraction means separating target material from bulk regolith,
processing it into a usable form, and doing this with minimal
mass launched from Earth.

Three approaches dominate current research:

**1. Thermal extraction (volatiles from C-types)**

- Focus sunlight onto regolith using a concentrator mirror
- Heat to 200--400 degrees Celsius; water vapour is driven off
- Capture in a cold trap; condense; split by electrolysis into H2 and O2
- Pro: simple process, well-understood chemistry, no mechanical complexity
- Con: C-type regolith is only ~5--10 wt% water at best;
  large volumes of regolith must be processed

**2. Magnetic separation (metals from S and M-types)**

- Crush or grind regolith to liberate iron-nickel grains from silicate matrix
- Apply magnetic field to separate metal from the rest
- Pro: no heat required; iron-nickel is strongly magnetic
- Con: crushing and grinding in microgravity is an unsolved engineering problem;
  fragment and dust management in near-zero gravity is poorly understood

**3. Electrolysis of extracted water**

- Input: water extracted by thermal method
- Output: H2 (fuel) + O2 (oxidiser)
- This is the same propellant pair as the Space Launch System upper stage
- The process is well-proven on Earth and demonstrated in small ISS experiments
- Challenge: electrode lifetime and power supply at scale in deep space


### C1. Mass balance: thermal extraction from a C-type asteroid

Adjust the parameters and re-run to explore different scenarios.


In [ ]:
# ---- Adjust these ----
asteroid_diameter_m   = 50.0    # metres
water_content_wt_pct  = 5.0     # typical CI: ~10%, CM: ~5%
extraction_efficiency = 0.50    # fraction of water actually captured
bulk_density_kg_m3    = 1300    # C-type porous bulk density
# ----------------------

volume_m3   = (4/3) * np.pi * (asteroid_diameter_m / 2) ** 3
mass_t      = volume_m3 * bulk_density_kg_m3 / 1000
water_t     = mass_t * (water_content_wt_pct / 100)
extracted_t = water_t * extraction_efficiency

# Electrolysis stoichiometry: 9 kg water -> 1 kg H2 + 8 kg O2
h2_t   = extracted_t / 9
o2_t   = extracted_t * 8 / 9
prop_t = h2_t + o2_t

print('===== Thermal extraction estimate =====')
print(f'Asteroid diameter:           {asteroid_diameter_m:.0f} m')
print(f'Volume:                      {volume_m3:,.0f} m3')
print(f'Total mass:                  {mass_t:,.0f} tonnes')
print(f'Water in asteroid:           {water_t:,.0f} tonnes')
print(f'Extraction efficiency:       {extraction_efficiency*100:.0f}%')
print(f'Water extracted:             {extracted_t:,.0f} tonnes')
print()
print('After electrolysis:')
print(f'  H2 (fuel):                 {h2_t:,.0f} tonnes')
print(f'  O2 (oxidiser):             {o2_t:,.0f} tonnes')
print(f'  Total propellant:          {prop_t:,.0f} tonnes')
print()
print(f'  Falcon 9 upper stage uses  ~92 t propellant')
print(f'  Starship uses              ~1,200 t propellant')
print(f'  Equivalent Falcon 9 loads: {prop_t/92:.1f}')
print()
print('Upper bound only -- thermal losses and infrastructure mass not included.')


### C2. Extraction yield vs asteroid size


In [ ]:
c_density, c_water, c_eff = 1300, 0.05, 0.50
s_density, s_metal, s_eff = 3500, 0.15, 0.40

diams_log = np.logspace(0.8, 3.2, 80)

water_yield = [
    (4/3)*np.pi*(d/2)**3 * c_density/1000 * c_water * c_eff
    for d in diams_log
]
metal_yield = [
    (4/3)*np.pi*(d/2)**3 * s_density/1000 * s_metal * s_eff
    for d in diams_log
]

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(diams_log, water_yield, color=TEAL, linewidth=2,
          label='Water from C-type (tonnes)')
ax.loglog(diams_log, metal_yield, color=ROSE, linewidth=2,
          label='Iron-nickel from S-type (tonnes)')

for y_val, label in [(92,   'Falcon 9 upper stage (~92 t)'),
                      (1200, 'Starship (~1,200 t)')]:
    ax.axhline(y_val, color=SAND, linestyle=':', linewidth=0.9, alpha=0.7)
    ax.text(1.1, y_val * 1.4, label, color=SAND, fontsize=8)

ax.set_xlabel('Asteroid diameter (m)')
ax.set_ylabel('Extracted mass (tonnes)')
ax.set_title(
    f'Extraction yield vs asteroid size\n'
    f'C-type: {c_density} kg/m3, {c_water*100:.0f} wt% water, {c_eff*100:.0f}% extraction   |   '
    f'S-type: {s_density} kg/m3, {s_metal*100:.0f} wt% metal, {s_eff*100:.0f}% separation',
    color=GOLD, fontsize=10)
ax.legend(fontsize=9)
ax.grid(True, which='both')
plt.tight_layout()
plt.show()


### C3. The energy problem

Extraction is not just a mass problem -- it is an energy problem.

| Method | Energy per tonne processed | Supply approach |
|---|---|---|
| Thermal extraction | ~500 MJ per tonne of regolith | Solar concentrator mirror |
| Electrolysis | ~160 MJ per tonne of water | Solar photovoltaic array |
| Magnetic separation | ~50 MJ per tonne of metal | Solar photovoltaic array |

At 1 AU, solar irradiance is ~1360 W/m2.
A 100 m2 array at 20% efficiency produces ~27 kW.
Processing one tonne of C-type regolith thermally takes that array about 5 hours.

A 50 m diameter asteroid contains ~85,000 tonnes of regolith.
Full thermal processing at that power level would take decades.
In practice, early missions would target surface or near-surface material only.

The startup companies Planetary Resources and Deep Space Industries both attempted
asteroid mining in the 2010s and failed commercially, despite viable physics.
The gap between a physically sound concept and a profitable business
remains the central challenge for the industry.


---
## Workshop complete

**Key ideas**

- NEAs are a small, dynamically perturbed population originating in the Main Belt.
  They are targets for resource missions because their orbits require far less
  delta-v to reach than the Main Belt or the lunar surface.
- Low delta-v and low MOID often coincide. The easiest-to-reach mining targets
  overlap with the asteroid population monitored most carefully for impact risk.
- C-type asteroids carry water as phyllosilicates; S-types carry iron-nickel metal.
  Most accessible NEAs have unknown composition.
- Thermal extraction of water from C-types is the most mature extraction concept.
  The physics is sound; the economics are not yet viable.

**Further reading**

- DeMeo & Carry (2014). Solar System evolution from compositional mapping.
  *Nature* 505, 629--634.
- Watanabe et al. (2019). Hayabusa2 arrives at Ryugu. *Science* 364, 268--272.
- Dunn et al. (2010). Mineralogy of ordinary chondrite parent bodies. *Icarus* 208, 789--797.
- Elvis, M. (2014). How many ore-bearing asteroids?
  *Planetary and Space Science* 91, 20--26.
- Granvik et al. (2018). Debiased orbit and absolute-magnitude distributions for NEOs.
  *Icarus* 312, 181--237.
- CNEOS NHATS: https://cneos.jpl.nasa.gov/nhats/
- JPL SBDB: https://ssd.jpl.nasa.gov/tools/sbdb_query.html
